# Libya Length of Growing Period (LGP) — Python / Earth Engine

Reproducible Python port of the Libya LGP workflow. Heavy raster computation remains server-side in Google Earth Engine.

**Independent personal project by Hamed Sabzchi Dehkharghani.**


In [ ]:
!pip -q install earthengine-api geemap ipywidgets pandas

import ee
PROJECT_ID = "practical-proxy-441422-n6"
ee.Authenticate(auth_mode="notebook", force=True)
ee.Initialize(project=PROJECT_ID)
print("SUCCESS: Earth Engine initialized.")


In [ ]:
import importlib.util, urllib.request, pathlib, sys

ENGINE_URL = "https://raw.githubusercontent.com/hamedsabzchi/libya-length-of-growing-period/main/python/lgp_engine.py"
engine_path = pathlib.Path("lgp_engine.py")
urllib.request.urlretrieve(ENGINE_URL, engine_path)
spec = importlib.util.spec_from_file_location("lgp_engine", engine_path)
lgp = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = lgp
spec.loader.exec_module(lgp)
print("SUCCESS: LGP engine loaded from GitHub.")


## Quick validation map — no button required

Run the next cell. It directly builds and displays a lightweight validation scenario for growing years 2022–2024. The fixed 100 mm storage option is used here only to keep the first live test simple.


In [ ]:
import geemap

quick_config = lgp.LGPConfig(
    start_year=2022,
    end_year=2024,
    rainfall="CHIRPS v2 Daily",
    et="TerraClimate PET",
    temperature="TerraClimate temperature",
    method="Gintzburger and Saidi method",
    storage="Reference storage 100 mm",
    reference_storage_mm=100,
)

lgp.validate_config(quick_config)
quick_result = lgp.build_result(quick_config)
print("SUCCESS: LGP computation graph created.")
print(lgp.configuration_summary(quick_config))

quick_map = geemap.Map(center=[27.0, 17.0], zoom=5)
quick_map.addLayer(
    quick_result["classes"],
    {"min": 1, "max": 5, "palette": lgp.CLASS_PALETTE},
    "LGP Classification 2022-2024",
)
quick_map.addLayer(
    lgp.libya_boundary().style(color="111111", fillColor="00000000", width=2),
    {},
    "Libya Boundary",
)
quick_map


## Optional interactive controls

After the direct validation map works, the controls below can be used for other supported configurations.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd

period = widgets.Dropdown(options=[("2020-2024", (2020, 2024)), ("2021-2025", (2021, 2025)), ("1991-2020", (1991, 2020)), ("1982-2024", (1982, 2024))], value=(2020, 2024), description="Period")
rain = widgets.Dropdown(options=lgp.RAINFALL_SOURCES, value="CHIRPS v2 Daily", description="Rainfall")
et = widgets.Dropdown(options=lgp.ET_SOURCES, value="TerraClimate PET", description="ET")
temp = widgets.Dropdown(options=lgp.TEMPERATURE_SOURCES, value="TerraClimate temperature", description="Temp")
method = widgets.Dropdown(options=lgp.METHODS, value="Gintzburger and Saidi method", description="Method")
storage = widgets.Dropdown(options=lgp.STORAGE_SOURCES, value="SoilGrids spatial WR 0-60 cm", description="Storage")
wr = widgets.FloatSlider(value=100, min=5, max=150, step=5, description="WR mm")
areas = widgets.Checkbox(value=False, description="Area statistics")
run_btn = widgets.Button(description="GENERATE MAP", button_style="success")
export_btn = widgets.Button(description="Export GeoTIFF to Drive")
out = widgets.Output()

state = {"result": None, "config": None}

def make_config():
    start_year, end_year = period.value
    return lgp.LGPConfig(start_year=start_year, end_year=end_year, rainfall=rain.value, et=et.value, temperature=temp.value, method=method.value, storage=storage.value, reference_storage_mm=wr.value)

def run_analysis(_=None):
    with out:
        clear_output(wait=True)
        try:
            config = make_config()
            lgp.validate_config(config)
            result = lgp.build_result(config)
            state["result"] = result
            state["config"] = config
            print(lgp.configuration_summary(config))
            m = geemap.Map(center=[27.0, 17.0], zoom=5)
            m.addLayer(result["classes"], {"min": 1, "max": 5, "palette": lgp.CLASS_PALETTE}, "LGP Classification")
            m.addLayer(lgp.libya_boundary().style(color="111111", fillColor="00000000", width=2), {}, "Libya Boundary")
            display(m)
            if areas.value:
                fc = lgp.area_statistics(result["classes"], lgp.effective_scale(config.rainfall, config.et, config.temperature))
                rows = fc.getInfo().get("features", [])
                df = pd.DataFrame([f["properties"] for f in rows])
                display(df.sort_values("class"))
        except Exception as exc:
            print("ERROR:", exc)

def export_result(_=None):
    with out:
        if not state["result"]:
            print("Generate a map first.")
            return
        task = lgp.start_drive_export(state["result"]["classes"], state["config"])
        print("Export task started:", task.id)

run_btn.on_click(run_analysis)
export_btn.on_click(export_result)
display(widgets.VBox([period, rain, et, temp, method, storage, wr, areas, widgets.HBox([run_btn, export_btn]), out]))


## Interpretation

The final map is the classification of the **pixel-wise median annual LGP** over the selected period. Each annual growing year runs from November of the previous calendar year through October of the named year. Approximate LGP days are computed as median qualifying months × 30.

This is a climatic LGP indicator, not a complete crop-suitability or national agro-ecological zoning assessment.
